comparing the wta and std single particle distributions. The 2D single particle distributions are already saved in the yield histogram root files, this file projects those 2D distributions onto $\eta *$

In [2]:
import ROOT
import numpy as np
import math
import pandas as pd
import fastjet
import matplotlib.pyplot as plt
import os
import ctypes
import array

In [10]:
#input_file = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/3mb_nch60/merged/merged_signals.root"
#output_dir_raw = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/comparing_frames/EPD_proj/raw"
#output_dir_norm = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/comparing_frames/EPD_proj/normalised"

In [11]:
#input_file = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/from_NOTS/0mb_nch60/Merged_Signals.root"
#output_dir_raw = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/comparing_frames/EPD_proj/0mb/raw"
#output_dir_norm = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/comparing_frames/EPD_proj/0mb/normalised"

In [3]:
input_file = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/3mb_nch60/Merged_Signals.root"
output_dir_raw = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/3mb/raw"
output_dir_norm = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/3mb/normalised"
output_dir_norm_nch = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/3mb/normalised_nch"

In [22]:
# 0mb files:

input_file = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/from_NOTS/0mb_nch60/Merged_Signals.root"
output_dir_raw = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/raw"
output_dir_norm = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised"
output_dir_norm_nch = "/Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised_nch"

In [23]:
def openHists(multbin):
    '''
    open the EPD histograms for a given mult bin
    '''
    file = ROOT.TFile.Open(input_file, "READ")

    bin_name = f"{multbin[0]}_{multbin[1]}"

    hEPD_wta = file.Get(f"WTA_epd_{bin_name}")
    hEPD_std = file.Get(f"STD_epd_{bin_name}")

    epd_wta = hEPD_wta.Clone()
    epd_std = hEPD_std.Clone()

    epd_wta.SetDirectory(0)
    epd_std.SetDirectory(0)

    total_Nch_wta = file.Get(f"total_Nch_WTA_{bin_name}").GetVal()
    total_Nch_std = file.Get(f"total_Nch_STD_{bin_name}").GetVal()

    file.Close()

    return epd_wta, epd_std, total_Nch_wta, total_Nch_std


In [24]:
def project(hEPD, suffix):
    '''
    Makes the 1d projection of the epd histograms
    input: epd histogram
    output: 1d projection onto eta star
    '''
    
    # doing the projection onto eta star
    h1D = hEPD.Clone()
    h1D = h1D.ProjectionX(f"h1D_{suffix}")

    return h1D

In [25]:
def normaliseProjection(h1D_wta, h1D_std):
    '''
    Normalises the projection onto eta star by the total number of particles in the histogram 
    (doing this because number of particles is different for WTA and std axes)
    inputs: 
        - h1D_wta, h1D_std: the 1D histograms
    output: normalised 1D projection onto eta star
    '''
    integral_wta = h1D_wta.Integral()
    integral_std = h1D_std.Integral()

    if integral_wta != 0:
        h1D_wta.Scale(100.0 / integral_wta)
    if integral_std !=0:
        h1D_std.Scale(100.0 / integral_std)

In [26]:
def normaliseProjection_by_Nch(h1D_wta, h1D_std, Nch_wta, Nch_std):
    '''
    Normalises by total Nch for the bin instead of by graph area
    This way normalisation takes into account particles that failed jet frame cuts
    '''
    if Nch_wta > 0:
        h1D_wta.Scale(100.0 / Nch_wta)
    if Nch_std >0:
        h1D_std.Scale(100.0 / Nch_std)


In [27]:
def saveProjections(mult_bin, h1D_wta, h1D_std):
    '''
    saves the projection histograms
    '''

    # Create the folder if it doesn't already exist
    os.makedirs(output_dir, exist_ok=True)

    # 2. Create the dynamic filename
    if len(mult_bin) == 2:
        bin_name = f"{mult_bin[0]}_{mult_bin[1]}"
    else:
        bin_name = f"{mult_bin[0]}_up"

    filename = f"EPD_Projection_Histograms_Mult_{bin_name}.root"

    # 3. Combine the folder path and the filename
    full_filepath = os.path.join(output_dir, filename)

    # 4. Open the ROOT file using the FULL path
    out_file = ROOT.TFile(full_filepath, "RECREATE")

    # 5. Rename and write the histograms
    h1D_wta.SetName(f"WTA_projection_{bin_name}")
    h1D_std.SetName(f"STD_projection_{bin_name}")

    h1D_wta.Write()
    h1D_std.Write()

    out_file.Close()

    print(f"Successfully saved projections to {full_filepath}")

In [28]:
ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0) # Turn off the default statistical info boxes

def draw_projection(h1D_wta, h1D_std, mult_bin, norm, nch):
    """
    Draws a 1D eta star projection histogram

    inputs:
        - norm: true if normalised, false if not
        - nch: true if normalised by Nch, false if normalised by area
    """

    # make the file name
    if len(mult_bin) == 2:
        bin_name = f"{mult_bin[0]}_{mult_bin[1]}"
    else:
        bin_name = f"{mult_bin[0]}_up"

    if norm:
        if nch:
            filename = f"normalised_by_Nch_EPD_Projection_Mult_{bin_name}.pdf"
            save_path = os.path.join(output_dir_norm_nch, filename)
        else:
            filename = f"normalised_EPD_Projection_Mult_{bin_name}.pdf"
            save_path = os.path.join(output_dir_norm, filename)
    else:
        filename = f"EPD_Projection_Mult_{bin_name}.pdf"
        save_path = os.path.join(output_dir_raw, filename)

    canvas = ROOT.TCanvas("c_proj", "EPD Projection", 800, 600)
    canvas.SetLeftMargin(0.15)
    canvas.SetBottomMargin(0.15)
    
    # General histogram aesthetics
    h1D_wta.SetMarkerStyle(20) # Filled circle
    h1D_wta.SetMarkerSize(1.1)
    h1D_wta.SetMarkerColor(ROOT.kBlue)
    h1D_wta.SetLineColor(ROOT.kBlue)
    h1D_wta.SetStats(0)

    h1D_std.SetMarkerStyle(25) 
    h1D_std.SetMarkerSize(1.1)
    h1D_std.SetMarkerColor(ROOT.kRed)
    h1D_std.SetLineColor(ROOT.kRed)
    h1D_std.SetStats(0)

    # 3. Dynamic Y-Axis Scaling
    # Find the global max and min so neither graph gets clipped
    max_y = max(h1D_wta.GetMaximum(), h1D_std.GetMaximum())
    min_y = min(h1D_wta.GetMinimum(), h1D_std.GetMinimum())
    
    # Add 25% headroom to the top for the legend
    h1D_wta.SetMaximum(max_y + 0.25 * abs(max_y))
    h1D_wta.SetMinimum(min_y - 0.10 * abs(min_y))
    h1D_wta.GetXaxis().SetRangeUser(0, 5.5)
    
    # --- ROOT TLatex Title Fixes ---
    if norm:
        if nch:
            y_axis_title = "% of N_{ch}"
        else:
            y_axis_title = f"% of particles"
    elif not norm:
        y_axis_title = "Number of particles"
    
    if len(mult_bin) > 1:
        h1D_wta.SetTitle(f"EPD projection for multiplicity Bin: {mult_bin[0]} #leq N_{{ch}} < {mult_bin[1]}; #eta*; {y_axis_title}")
    elif len(mult_bin) == 1:
        h1D_wta.SetTitle(f"EPD projection for multiplicity Bin: {mult_bin[0]} #leq N_{{ch}}; #eta*; {y_axis_title}")

    # Draw 
    h1D_wta.Draw("HIST E PL") # P = Markers, 
    h1D_std.Draw("HIST E PL SAME")
        
    
    # 6. Add a standard Legend
    legend = ROOT.TLegend(0.45, 0.75, 0.88, 0.88)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0) # Transparent background
    legend.SetTextSize(0.035)
    legend.AddEntry(h1D_wta, "Winner-Take-All (WTA)", "pe")
    legend.AddEntry(h1D_std, "Standard Axis", "pe")
    legend.Draw()
    
    # 7. Save directly as a PDF
    canvas.Update()
    canvas.SaveAs(save_path)
    canvas.Close()


In [29]:
def clean_hist(hist_obj):
    '''
    Safely deletes the C++ underlying memory of a detached ROOT object
    '''
    if hist_obj:
        hist_obj.Delete()

In [30]:
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97, 1000] ]


In [31]:
# raw versions

for mult_bin in analysis_bins:
        
    hEPD_wta, hEPD_std, _, _ = openHists(mult_bin)

    h1D_wta = project(hEPD_wta, "wta")
    h1D_std = project(hEPD_std, "std")

    #draw the graphs
    draw_projection(h1D_wta, h1D_std, mult_bin, False, False)

    #clean out the histograms
    clean_hist(hEPD_wta)
    clean_hist(hEPD_std)
    clean_hist(h1D_wta)
    clean_hist(h1D_std)
    ROOT.gDirectory.Clear()



Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/raw/EPD_Projection_Mult_0_25.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/raw/EPD_Projection_Mult_25_36.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/raw/EPD_Projection_Mult_36_48.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/raw/EPD_Projection_Mult_48_60.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/raw/EPD_Projection_Mult_60_71.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/roha

In [32]:
# making the area normalised versions

for mult_bin in analysis_bins:
        
    hEPD_wta, hEPD_std, _ , _ = openHists(mult_bin)

    h1D_wta = project(hEPD_wta, "wta")
    h1D_std = project(hEPD_std, "std")

    normaliseProjection(h1D_wta, h1D_std)

    #draw the graphs
    draw_projection(h1D_wta, h1D_std, mult_bin, True, False)

    #clean out the histograms
    clean_hist(hEPD_wta)
    clean_hist(hEPD_std)
    clean_hist(h1D_wta)
    clean_hist(h1D_std)
    ROOT.gDirectory.Clear()

Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised/normalised_EPD_Projection_Mult_0_25.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised/normalised_EPD_Projection_Mult_25_36.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised/normalised_EPD_Projection_Mult_36_48.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised/normalised_EPD_Projection_Mult_48_60.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised/normalised_EPD

In [33]:
# making the normalised by Nch versions

for mult_bin in analysis_bins:
        
    hEPD_wta, hEPD_std, Nch_wta, Nch_std = openHists(mult_bin)

    h1D_wta = project(hEPD_wta, "wta")
    print(f"WTA {mult_bin} integral:", h1D_wta.Integral())
    h1D_std = project(hEPD_std, "std")
    print(f"STD {mult_bin} integral:", h1D_std.Integral())

    normaliseProjection_by_Nch(h1D_wta, h1D_std, Nch_wta, Nch_std)

    #draw the graphs
    draw_projection(h1D_wta, h1D_std, mult_bin, True, True)

    #clean out the histograms
    clean_hist(hEPD_wta)
    clean_hist(hEPD_std)
    clean_hist(h1D_wta)
    clean_hist(h1D_std)
    ROOT.gDirectory.Clear()

WTA [0, 25] integral: 0.0
STD [0, 25] integral: 0.0
WTA [25, 36] integral: 0.0
STD [25, 36] integral: 0.0
WTA [36, 48] integral: 0.0
STD [36, 48] integral: 0.0
WTA [48, 60] integral: 231.0
STD [48, 60] integral: 231.0
WTA [60, 71] integral: 1532533707.0
STD [60, 71] integral: 1630803516.0
WTA [71, 78] integral: 248658282.0
STD [71, 78] integral: 262970371.0
WTA [78, 91] integral: 103729806.0
STD [78, 91] integral: 109295901.0
WTA [91, 97] integral: 7943247.0
STD [91, 97] integral: 8335748.0
WTA [97, 1000] integral: 4108470.0
STD [97, 1000] integral: 4300689.0


Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised_nch/normalised_by_Nch_EPD_Projection_Mult_0_25.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised_nch/normalised_by_Nch_EPD_Projection_Mult_25_36.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised_nch/normalised_by_Nch_EPD_Projection_Mult_36_48.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_frames/EPD_proj/0mb/normalised_nch/normalised_by_Nch_EPD_Projection_Mult_48_60.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/rohanjagadeesan/Desktop/Code/li_lab/WinnerTakeAll/output/new_radius/comparing_f